##  config

In [ ]:
import os
import sys
from pathlib import Path

ROOT = "/home/wangxc1117/STDK_GNA_Research"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import optuna
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

torch.set_default_dtype(torch.float32)
optuna.logging.set_verbosity(optuna.logging.WARNING)

from examples.baselines.stdk.st_interp import create_model
from spatial_adapter.models.spatial_adapter import (
    SpatialNeuralAdapterConfig,
    ADMMConfig,
    TrainingConfig,
    BasisConfig,
)
from examples.baselines.timesplit.experiment_core import (
    seed_everything,
    build_fixed_location_subset,
    build_contiguous_time_splits,
    build_stdk_model_config,
    train_simple_loop,
    predict_all_simple,
    rmse_pooled,
    rmse_on_time_subset,
    DictDataset,
    collate_fn,
    new_trend_basis,
    fit_adapter_reconstruct_all_times,
    plot_trial_maps,
    fmt_pm,
    build_standard_bin_edges,
    sv_match_loss_for_prediction_matrix,
)
from spatial_adapter.cpp_extensions import spatial_utils


# Global settings & dirs
SEED = 123

WEATHER2K_NPY = Path("/home/wangxc1117/Weather2K/weather2k.npy")
TARGET_VAR_IDX = 4
LAT_IDX = 0
LON_IDX = 1
T_KEEP = 1000

EPOCHS = 350
BATCH_SIZE = 512
LR = 1e-3
WEIGHT_DECAY = 1e-5

SPACE_RATIO_KEEP = 0.1
HELDOUT_STATION_RATIO = 0.2

TRAIN_RATIO_TIME = 0.1
VAL_RATIO_TIME = 0.1
TEST_RATIO_TIME = 0.8

GNA_BATCH_SIZE = 64
N_TRIALS = 50
TAU_MIN = 1e-8
TAU_MAX = 1e4

K_FIXED = 40
N_RUNS_FIXED_K = 10

SEMIVAR_WEIGHTED = True
SEMIVAR_NORMALIZED = False
SEMIVAR_ESTIMATOR = "matheron"

TUNING_TARGET = "rmse"

if TUNING_TARGET not in {"rmse", "covfrob", "sv_score"}:
    raise ValueError("TUNING_TARGET must be one of {'rmse', 'covfrob', 'sv_score'}")

RESULT_DIR = Path("./weather2k")
REPEAT_DIR = RESULT_DIR / (
    f"{TUNING_TARGET}_tuning"
    f"/var{TARGET_VAR_IDX}_tkeep{T_KEEP}"
    f"_k_{K_FIXED}_fixedspace{SPACE_RATIO_KEEP}"
    f"_time_train{TRAIN_RATIO_TIME}_val{VAL_RATIO_TIME}_test{TEST_RATIO_TIME}"
)
REPEAT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = REPEAT_DIR / f"k_{K_FIXED}_repeat_runs_summary.csv"

PRED_DIR = REPEAT_DIR / "saved_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_DIR = REPEAT_DIR / "trial_results"
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

PHI_DIR = REPEAT_DIR / "saved_phi"
PHI_DIR.mkdir(parents=True, exist_ok=True)

ALL_TRIAL_CSV = TRIAL_DIR / "all_trial_results.csv"

DIAG_DIR = REPEAT_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device, flush=True)

seed_everything(SEED)

config = SpatialNeuralAdapterConfig(
    admm=ADMMConfig(
        rho=1.0,
        dual_momentum=0.2,
        max_iters=3000,
        min_outer=20,
        tol=1e-4,
    ),
    training=TrainingConfig(
        lr_mu=1e-2,
        batch_size=GNA_BATCH_SIZE,
        pretrain_epochs=5,
    ),
    basis=BasisConfig(
        phi_every=5,
        phi_freeze=200,
    ),
)


# Data helpers
def load_weather2k_as_long_df(
    npy_path,
    target_var_idx,
    lat_idx,
    lon_idx,
    t_keep=None,
    normalize_xy=True,
):
    arr = np.load(str(npy_path)).astype(np.float32)

    if arr.ndim != 3:
        raise ValueError(f"Expected arr.ndim == 3 (S, V, T), got shape={arr.shape}")

    n_sites, n_vars, n_times_full = arr.shape

    if not (0 <= lat_idx < n_vars and 0 <= lon_idx < n_vars and 0 <= target_var_idx < n_vars):
        raise ValueError(
            f"Bad index: lat_idx={lat_idx}, lon_idx={lon_idx}, "
            f"target_var_idx={target_var_idx}, n_vars={n_vars}"
        )

    lat = arr[:, lat_idx, 0].astype(np.float32)
    lon = arr[:, lon_idx, 0].astype(np.float32)
    z_full = arr[:, target_var_idx, :].astype(np.float32)

    if t_keep is not None:
        if t_keep <= 0 or t_keep > n_times_full:
            raise ValueError(f"t_keep must be in [1, {n_times_full}], got {t_keep}")
        z_use = z_full[:, -t_keep:]
        n_times = t_keep
    else:
        z_use = z_full
        n_times = n_times_full

    if normalize_xy:
        lon_min, lon_max = float(np.min(lon)), float(np.max(lon))
        lat_min, lat_max = float(np.min(lat)), float(np.max(lat))
        x = ((lon - lon_min) / (lon_max - lon_min + 1e-12)).astype(np.float32)
        y = ((lat - lat_min) / (lat_max - lat_min + 1e-12)).astype(np.float32)
    else:
        x = lon.astype(np.float32)
        y = lat.astype(np.float32)

    t = np.arange(n_times, dtype=np.int64)

    xx = np.repeat(x, n_times)
    yy = np.repeat(y, n_times)
    tt = np.tile(t, n_sites)
    zz = z_use.reshape(-1).astype(np.float32)

    df = pd.DataFrame({
        "x": xx,
        "y": yy,
        "t": tt,
        "z": zz,
    })

    z_np = df["z"].to_numpy(np.float32)
    ok = np.isfinite(z_np)
    df = df.loc[ok].reset_index(drop=True)

    return df


# Metric helpers
# Objective helpers
def choose_objective_value(val_rmse, covfrob_reg_val, sv_loss_val):
    if TUNING_TARGET == "rmse":
        return float(val_rmse)
    if TUNING_TARGET == "covfrob":
        return float(covfrob_reg_val)
    if TUNING_TARGET == "sv_score":
        return float(sv_loss_val)
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")


def objective_label():
    if TUNING_TARGET == "rmse":
        return "best_val_rmse"
    if TUNING_TARGET == "covfrob":
        return "best_val_covfrob"
    if TUNING_TARGET == "sv_score":
        return "best_val_sv_score"
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")


def objective_best_by_column():
    if TUNING_TARGET == "rmse":
        return "val_rmse_raw"
    if TUNING_TARGET == "covfrob":
        return "covfrob_reg_val"
    if TUNING_TARGET == "sv_score":
        return "sv_loss_val"
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")


# IO helpers
def load_all_trials_or_empty():
    if ALL_TRIAL_CSV.exists():
        return pd.read_csv(ALL_TRIAL_CSV)
    return pd.DataFrame()


def load_summary_or_empty():
    if SUMMARY_CSV.exists():
        return pd.read_csv(SUMMARY_CSV)
    return pd.DataFrame()


def list_prediction_npz_files():
    return sorted(PRED_DIR.glob("seed_*.npz"))


def print_main_paths():
    print("REPEAT_DIR:", REPEAT_DIR, flush=True)
    print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)
    print("ALL_TRIAL_CSV:", ALL_TRIAL_CSV, flush=True)
    print("PRED_DIR:", PRED_DIR, flush=True)
    print("TRIAL_DIR:", TRIAL_DIR, flush=True)
    print("PHI_DIR:", PHI_DIR, flush=True)
    print("DIAG_DIR:", DIAG_DIR, flush=True)


def split_observed_heldout_sites(df: pd.DataFrame, heldout_ratio: float, seed: int):
    df = df.copy()
    df["site_id"] = pd.factorize(list(zip(df["x"], df["y"])))[0]
    uniq_sites = np.sort(df["site_id"].unique())
    n_sites = len(uniq_sites)

    rs = np.random.RandomState(seed)
    perm = rs.permutation(n_sites)

    n_heldout = int(np.round(heldout_ratio * n_sites))
    n_heldout = max(1, min(n_heldout, n_sites - 1))

    heldout_sites = np.sort(perm[:n_heldout])
    observed_sites = np.sort(np.setdiff1d(uniq_sites, heldout_sites))

    df_obs = df[df["site_id"].isin(observed_sites)].copy().reset_index(drop=True)
    df_held = df[df["site_id"].isin(heldout_sites)].copy().reset_index(drop=True)

    return df_obs, df_held, observed_sites, heldout_sites


def subset_df_by_times(df: pd.DataFrame, times: np.ndarray) -> pd.DataFrame:
    return df[df["t"].isin(times)].copy().reset_index(drop=True)


def build_X_coords_t_y(df: pd.DataFrame):
    coords = df[["x", "y"]].to_numpy(np.float32)
    t = df["t_norm"].to_numpy(np.float32).reshape(-1, 1)
    y = df["z"].to_numpy(np.float32).reshape(-1, 1)
    X = np.empty((df.shape[0], 0), dtype=np.float32)
    return X, coords, t, y


def build_field_matrix_from_df_and_pred(df: pd.DataFrame, y_pred_flat: np.ndarray):
    coords_all = df[["x", "y"]].to_numpy(np.float32)
    uniq_t = np.sort(df["t"].unique())
    locs, inv_loc = np.unique(coords_all, axis=0, return_inverse=True)
    t_to_idx = {t: i for i, t in enumerate(uniq_t)}

    T = len(uniq_t)
    N = len(locs)

    t_idx = np.array([t_to_idx[t] for t in df["t"].to_numpy()])
    s_idx = inv_loc

    y_pred_mat = np.full((T, N), np.nan, np.float32)
    y_true_mat = np.full((T, N), np.nan, np.float32)

    y_pred_mat[t_idx, s_idx] = y_pred_flat.astype(np.float32)
    y_true_mat[t_idx, s_idx] = df["z"].to_numpy(np.float32)

    return y_true_mat, y_pred_mat, locs, uniq_t


def center_columns(mat: np.ndarray):
    col_mean = np.mean(mat, axis=0, keepdims=True)
    return mat - col_mean, col_mean

In [ ]:
from spatial_adapter.metrics import rmse_pooled, mae_pooled, r2_pooled, empirical_cov, cov_frob_observed


## seed runs

In [ ]:
def run_once_fixed_k(run_seed: int):
    seed_everything(run_seed)

    df_full = load_weather2k_as_long_df(
        npy_path=WEATHER2K_NPY,
        target_var_idx=TARGET_VAR_IDX,
        lat_idx=LAT_IDX,
        lon_idx=LON_IDX,
        t_keep=T_KEEP,
        normalize_xy=True,
    )

    df_run, keep_sites_run, n_sites_full_run = build_fixed_location_subset(
        df=df_full,
        keep_ratio=SPACE_RATIO_KEEP,
        seed=run_seed + 11111,
    )

    df_obs_run, df_held_run, observed_sites_run, heldout_sites_run = split_observed_heldout_sites(
        df=df_run,
        heldout_ratio=HELDOUT_STATION_RATIO,
        seed=run_seed + 22222,
    )

    df_obs_run["t_norm"] = (
        (df_obs_run["t"] - df_obs_run["t"].min())
        / (df_obs_run["t"].max() - df_obs_run["t"].min() + 1e-12)
    ).astype(np.float32)

    df_held_run["t_norm"] = (
        (df_held_run["t"] - df_obs_run["t"].min())
        / (df_obs_run["t"].max() - df_obs_run["t"].min() + 1e-12)
    ).astype(np.float32)

    (
        train_mask_flat_run,
        val_mask_flat_run,
        test_mask_flat_run,
        train_time_idx_run,
        val_time_idx_run,
        test_time_idx_run,
        uniq_t_run,
        n_times_run,
    ) = build_contiguous_time_splits(
        df=df_obs_run,
        train_ratio=TRAIN_RATIO_TIME,
        val_ratio=VAL_RATIO_TIME,
        test_ratio=TEST_RATIO_TIME,
    )

    df_obs_run["z_raw"] = df_obs_run["z"].astype(np.float32)

    z_train_raw = df_obs_run.loc[train_mask_flat_run, "z_raw"].to_numpy(np.float32)
    z_mean_run = float(np.mean(z_train_raw))
    z_sd_run = float(np.std(z_train_raw, ddof=0))
    if z_sd_run < 1e-12:
        z_sd_run = 1.0

    df_obs_run["z"] = (
        (df_obs_run["z_raw"].to_numpy(np.float32) - z_mean_run) / (z_sd_run + 1e-12)
    ).astype(np.float32)

    df_held_run["z_raw"] = df_held_run["z"].astype(np.float32)
    df_held_run["z"] = (
        (df_held_run["z_raw"].to_numpy(np.float32) - z_mean_run) / (z_sd_run + 1e-12)
    ).astype(np.float32)

    def to_raw(arr_std: np.ndarray) -> np.ndarray:
        return arr_std * z_sd_run + z_mean_run

    def mpiw(y_lower: np.ndarray, y_upper: np.ndarray, mask: np.ndarray) -> float:
        width = y_upper - y_lower
        return float(np.mean(width[mask]))

    def cp_percent(y_true: np.ndarray, y_lower: np.ndarray, y_upper: np.ndarray, mask: np.ndarray) -> float:
        covered = (y_true >= y_lower) & (y_true <= y_upper)
        return float(np.mean(covered[mask]) * 100.0)

    coords_all_run = df_obs_run[["x", "y"]].to_numpy(np.float32)
    t_all_run = df_obs_run["t_norm"].to_numpy(np.float32).reshape(-1, 1)
    y_all_run = df_obs_run["z"].to_numpy(np.float32).reshape(-1, 1)
    X_all_run = np.empty((df_obs_run.shape[0], 0), dtype=np.float32)

    X_train_run = X_all_run[train_mask_flat_run]
    coords_train_run = coords_all_run[train_mask_flat_run]
    t_train_run = t_all_run[train_mask_flat_run]
    y_train_run = y_all_run[train_mask_flat_run]

    train_dataset_run = DictDataset(
        torch.from_numpy(X_train_run),
        torch.from_numpy(coords_train_run),
        torch.from_numpy(t_train_run),
        torch.from_numpy(y_train_run),
    )

    g = torch.Generator()
    g.manual_seed(run_seed + 1000)

    train_loader_run = DataLoader(
        train_dataset_run,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    stdk_config = build_stdk_model_config(
        EPOCHS=EPOCHS,
        LR=LR,
        WEIGHT_DECAY=WEIGHT_DECAY,
        BATCH_SIZE=BATCH_SIZE,
    )

    stdk_run = create_model(
        stdk_config,
        train_coords=coords_train_run,
    ).to(device)

    stdk_run = train_simple_loop(
        model=stdk_run,
        train_loader=train_loader_run,
        device=device,
        config=stdk_config,
    )

    y_hat_all_run = predict_all_simple(
        model=stdk_run,
        X=torch.from_numpy(X_all_run),
        coords=torch.from_numpy(coords_all_run),
        t=torch.from_numpy(t_all_run),
        batch_size=BATCH_SIZE,
        device=device,
    )

    locs_run, inv_loc_run = np.unique(coords_all_run, axis=0, return_inverse=True)
    t_to_idx_run = {t: i for i, t in enumerate(uniq_t_run)}

    T_run = len(uniq_t_run)
    N_run = len(locs_run)

    t_idx_run = np.array([t_to_idx_run[t] for t in df_obs_run["t"].to_numpy()])
    s_idx_run = inv_loc_run

    y_stdk_run = np.full((T_run, N_run), np.nan, np.float32)
    y_true_run = np.full((T_run, N_run), np.nan, np.float32)

    y_stdk_run[t_idx_run, s_idx_run] = y_hat_all_run
    y_true_run[t_idx_run, s_idx_run] = df_obs_run["z"].to_numpy(np.float32)

    y_stdk_raw_run = to_raw(y_stdk_run)
    y_true_raw_run = to_raw(y_true_run)

    residual_true_run = y_true_run - y_stdk_run

    time_feat_run = (
        (uniq_t_run - uniq_t_run.min())
        / (uniq_t_run.max() - uniq_t_run.min() + 1e-12)
    ).astype(np.float32)

    cont_all_run = (
        torch.from_numpy(time_feat_run)
        .float()
        .unsqueeze(1)
        .repeat(1, N_run)
        .unsqueeze(-1)
    )

    train_cont_train_run = cont_all_run[train_time_idx_run]
    train_y_train_run = torch.from_numpy(
        residual_true_run[train_time_idx_run, :]
    ).float()
    train_idx_train_run = torch.arange(len(train_time_idx_run), dtype=torch.long)

    gna_loader_train_run = DataLoader(
        TensorDataset(train_idx_train_run, train_cont_train_run, train_y_train_run),
        batch_size=min(GNA_BATCH_SIZE, len(train_time_idx_run)),
        shuffle=True,
        drop_last=False,
    )

    residual_true_tensor_all_run = torch.from_numpy(residual_true_run).float()

    seed_phi_dir = PHI_DIR / f"seed_{run_seed}"
    seed_phi_dir.mkdir(parents=True, exist_ok=True)

    semivar_bin_edges = build_standard_bin_edges(locs_run)
    full_time_idx_run = np.arange(T_run, dtype=int)

    def rmse_std(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        mask = np.isfinite(y_true_run[time_idx, :]) & np.isfinite(y_pred_std[time_idx, :])
        return rmse_pooled(y_true_run[time_idx, :], y_pred_std[time_idx, :], mask)

    def rmse_raw(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        y_pred_raw = to_raw(y_pred_std[time_idx, :])
        mask = np.isfinite(y_true_raw_run[time_idx, :]) & np.isfinite(y_pred_raw)
        return rmse_pooled(y_true_raw_run[time_idx, :], y_pred_raw, mask)

    def covfrob_std(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        return cov_frob_observed(
            y_true_run[time_idx, :],
            y_pred_std[time_idx, :],
        )

    def covfrob_raw(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        return cov_frob_observed(
            y_true_raw_run[time_idx, :],
            to_raw(y_pred_std[time_idx, :]),
        )

    def sv_loss_std(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        out = sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_pred_std,
            time_idx=time_idx,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )
        return float(out["loss"])

    def sv_loss_raw(y_pred_std: np.ndarray, time_idx: np.ndarray) -> float:
        out = sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_raw_run,
            y_pred=to_raw(y_pred_std),
            time_idx=time_idx,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )
        return float(out["loss"])

    writer_unreg = SummaryWriter(
        str(REPEAT_DIR / "logs_unreg" / f"seed_{run_seed}" / "unreg_tau1_0_tau2_0")
    )

    trend_unreg, basis_unreg = new_trend_basis(N_run, K_FIXED)
    fit_unreg = fit_adapter_reconstruct_all_times(
        tag="unreg_tau1_0_tau2_0",
        tau1=0.0,
        tau2=0.0,
        trend=trend_unreg,
        basis=basis_unreg,
        train_loader=gna_loader_train_run,
        val_cont=train_cont_train_run,
        val_y=train_y_train_run,
        locs=locs_run,
        config=config,
        device=device,
        cont_all=cont_all_run,
        residual_true_tensor_all=residual_true_tensor_all_run,
        writer=writer_unreg,
    )
    writer_unreg.close()

    residual_full_unreg_run = fit_unreg["pred_all"]
    diag_train_unreg_run = fit_unreg["diag_train"]
    diag_all_unreg_run = fit_unreg["diag_all"]
    phi_unreg_run = fit_unreg["phi"]

    np.savez_compressed(
        seed_phi_dir / "unreg_phi.npz",
        phi=phi_unreg_run.astype(np.float32),
        tau1=np.array([0.0], dtype=np.float64),
        tau2=np.array([0.0], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
    )

    y_final_unreg_run = y_stdk_run + residual_full_unreg_run
    y_final_unreg_raw_run = to_raw(y_final_unreg_run)

    trial_rows = []
    trial_cache = {}

    def objective(trial: optuna.Trial):
        tau1 = trial.suggest_float("tau1", TAU_MIN, TAU_MAX, log=True)
        tau2 = trial.suggest_float("tau2", TAU_MIN, TAU_MAX, log=True)

        tag = f"trial_{trial.number:03d}_tau1_{tau1:.3e}_tau2_{tau2:.3e}"
        writer_trial = SummaryWriter(
            str(REPEAT_DIR / "logs_reg" / f"seed_{run_seed}" / tag)
        )

        trend_trial, basis_trial = new_trend_basis(N_run, K_FIXED)
        fit_trial = fit_adapter_reconstruct_all_times(
            tag=tag,
            tau1=tau1,
            tau2=tau2,
            trend=trend_trial,
            basis=basis_trial,
            train_loader=gna_loader_train_run,
            val_cont=train_cont_train_run,
            val_y=train_y_train_run,
            locs=locs_run,
            config=config,
            device=device,
            cont_all=cont_all_run,
            residual_true_tensor_all=residual_true_tensor_all_run,
            writer=writer_trial,
        )
        writer_trial.close()

        residual_full_trial = fit_trial["pred_all"]
        diag_train_trial = fit_trial["diag_train"]
        diag_all_trial = fit_trial["diag_all"]
        phi_trial = fit_trial["phi"]

        y_final_trial = y_stdk_run + residual_full_trial

        train_rmse_std = rmse_std(y_final_trial, train_time_idx_run)
        val_rmse_std = rmse_std(y_final_trial, val_time_idx_run)
        test_rmse_std = rmse_std(y_final_trial, test_time_idx_run)
        full_rmse_std = rmse_std(y_final_trial, full_time_idx_run)

        train_rmse_raw = rmse_raw(y_final_trial, train_time_idx_run)
        val_rmse_raw = rmse_raw(y_final_trial, val_time_idx_run)
        test_rmse_raw = rmse_raw(y_final_trial, test_time_idx_run)
        full_rmse_raw = rmse_raw(y_final_trial, full_time_idx_run)

        covfrob_reg_train = covfrob_std(y_final_trial, train_time_idx_run)
        covfrob_reg_val = covfrob_std(y_final_trial, val_time_idx_run)
        covfrob_reg_test = covfrob_std(y_final_trial, test_time_idx_run)
        covfrob_reg_full = covfrob_std(y_final_trial, full_time_idx_run)

        covfrob_reg_train_raw = covfrob_raw(y_final_trial, train_time_idx_run)
        covfrob_reg_val_raw = covfrob_raw(y_final_trial, val_time_idx_run)
        covfrob_reg_test_raw = covfrob_raw(y_final_trial, test_time_idx_run)
        covfrob_reg_full_raw = covfrob_raw(y_final_trial, full_time_idx_run)

        sv_loss_train = sv_loss_std(y_final_trial, train_time_idx_run)
        sv_loss_val = sv_loss_std(y_final_trial, val_time_idx_run)
        sv_loss_test = sv_loss_std(y_final_trial, test_time_idx_run)
        sv_loss_full = sv_loss_std(y_final_trial, full_time_idx_run)

        sv_loss_train_raw = sv_loss_raw(y_final_trial, train_time_idx_run)
        sv_loss_val_raw = sv_loss_raw(y_final_trial, val_time_idx_run)
        sv_loss_test_raw = sv_loss_raw(y_final_trial, test_time_idx_run)
        sv_loss_full_raw = sv_loss_raw(y_final_trial, full_time_idx_run)

        objective_value = choose_objective_value(
            val_rmse=val_rmse_raw,
            covfrob_reg_val=covfrob_reg_val,
            sv_loss_val=sv_loss_val,
        )

        phi_path = seed_phi_dir / f"trial_{trial.number:03d}_phi.npz"
        np.savez_compressed(
            phi_path,
            phi=phi_trial.astype(np.float32),
            tau1=np.array([tau1], dtype=np.float64),
            tau2=np.array([tau2], dtype=np.float64),
            seed=np.array([run_seed], dtype=np.int32),
            trial=np.array([trial.number], dtype=np.int32),
        )

        row = {
            "seed": int(run_seed),
            "trial": int(trial.number),
            "tau1": float(tau1),
            "tau2": float(tau2),
            "log10_tau1": float(np.log10(tau1)),
            "log10_tau2": float(np.log10(tau2)),
            "objective_value": float(objective_value),
            "train_rmse": float(train_rmse_std),
            "val_rmse": float(val_rmse_std),
            "test_rmse": float(test_rmse_std),
            "full_rmse": float(full_rmse_std),
            "train_rmse_raw": float(train_rmse_raw),
            "val_rmse_raw": float(val_rmse_raw),
            "test_rmse_raw": float(test_rmse_raw),
            "full_rmse_raw": float(full_rmse_raw),
            "sv_loss_train": float(sv_loss_train),
            "sv_loss_val": float(sv_loss_val),
            "sv_loss_test": float(sv_loss_test),
            "sv_loss_full": float(sv_loss_full),
            "sv_loss_train_raw": float(sv_loss_train_raw),
            "sv_loss_val_raw": float(sv_loss_val_raw),
            "sv_loss_test_raw": float(sv_loss_test_raw),
            "sv_loss_full_raw": float(sv_loss_full_raw),
            "covfrob_reg_train": float(covfrob_reg_train),
            "covfrob_reg_val": float(covfrob_reg_val),
            "covfrob_reg_test": float(covfrob_reg_test),
            "covfrob_reg_full": float(covfrob_reg_full),
            "covfrob_reg_train_raw": float(covfrob_reg_train_raw),
            "covfrob_reg_val_raw": float(covfrob_reg_val_raw),
            "covfrob_reg_test_raw": float(covfrob_reg_test_raw),
            "covfrob_reg_full_raw": float(covfrob_reg_full_raw),
            "recon_mse_train": float(diag_train_trial["recon_mse"]),
            "smooth_penalty_train": float(diag_train_trial["smooth_penalty"]),
            "l1_penalty_train": float(diag_train_trial["l1_penalty"]),
            "total_surrogate_train": float(diag_train_trial["total_surrogate"]),
            "smooth_penalty_per_entry_train": float(diag_train_trial["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_train": float(diag_train_trial["l1_penalty_per_entry"]),
            "smooth_over_recon_train": float(diag_train_trial["smooth_over_recon"]),
            "l1_over_recon_train": float(diag_train_trial["l1_over_recon"]),
            "recon_mse_all": float(diag_all_trial["recon_mse"]),
            "smooth_penalty_all": float(diag_all_trial["smooth_penalty"]),
            "l1_penalty_all": float(diag_all_trial["l1_penalty"]),
            "total_surrogate_all": float(diag_all_trial["total_surrogate"]),
            "smooth_penalty_per_entry_all": float(diag_all_trial["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_all": float(diag_all_trial["l1_penalty_per_entry"]),
            "smooth_over_recon_all": float(diag_all_trial["smooth_over_recon"]),
            "l1_over_recon_all": float(diag_all_trial["l1_over_recon"]),
            "phi_path": str(phi_path),
        }

        trial_rows.append(row)
        trial_cache[int(trial.number)] = {
            "tau1": float(tau1),
            "tau2": float(tau2),
            "objective_value": float(objective_value),
            "pred_all": residual_full_trial.astype(np.float32),
            "phi": phi_trial.astype(np.float32),
            "diag_train": diag_train_trial,
            "diag_all": diag_all_trial,
        }

        print(
            f"[seed {run_seed}] trial {trial.number + 1:03d}/{N_TRIALS:03d} | "
            f"target={TUNING_TARGET} | "
            f"tau1={tau1:.3e} | tau2={tau2:.3e} | "
            f"train_rmse={train_rmse_std:.6f} | "
            f"val_rmse={val_rmse_std:.6f} | "
            f"sv_loss_val={sv_loss_val:.6f} | "
            f"covfrob_val={covfrob_reg_val:.6f} | "
            f"objective={objective_value:.6f}",
            flush=True,
        )

        return float(objective_value)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    trial_df_run = pd.DataFrame(trial_rows)
    trial_csv_run = TRIAL_DIR / f"seed_{run_seed}_trials.csv"
    trial_df_run.to_csv(trial_csv_run, index=False)

    best_trial_no = int(study.best_trial.number)
    best_trial = trial_cache[best_trial_no]

    best_tau1_run = float(best_trial["tau1"])
    best_tau2_run = float(best_trial["tau2"])
    best_objective_run = float(best_trial["objective_value"])
    residual_full_reg_best_run = best_trial["pred_all"]
    phi_reg_best_run = best_trial["phi"]
    diag_train_reg_best_run = best_trial["diag_train"]
    diag_all_reg_best_run = best_trial["diag_all"]

    np.savez_compressed(
        seed_phi_dir / "reg_best_phi.npz",
        phi=phi_reg_best_run.astype(np.float32),
        tau1=np.array([best_tau1_run], dtype=np.float64),
        tau2=np.array([best_tau2_run], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
        trial=np.array([best_trial_no], dtype=np.int32),
    )

    y_final_reg_best_run = y_stdk_run + residual_full_reg_best_run
    y_final_reg_best_raw_run = to_raw(y_final_reg_best_run)

    R_train_obs_run = residual_true_run[train_time_idx_run, :].astype(np.float64)
    R_train_obs_centered_run, R_train_col_mean_run = center_columns(R_train_obs_run)

    R_test_obs_run = residual_true_run[test_time_idx_run, :].astype(np.float64)
    R_test_obs_centered_run = R_test_obs_run - R_train_col_mean_run

    test_times_run = uniq_t_run[test_time_idx_run]
    df_held_test_run = subset_df_by_times(df_held_run, test_times_run)

    X_held_test_run, coords_held_test_run, t_held_test_run, y_held_test_run = build_X_coords_t_y(
        df=df_held_test_run
    )

    y_hat_held_test_run = predict_all_simple(
        model=stdk_run,
        X=torch.from_numpy(X_held_test_run),
        coords=torch.from_numpy(coords_held_test_run),
        t=torch.from_numpy(t_held_test_run),
        batch_size=BATCH_SIZE,
        device=device,
    )

    y_true_held_test_run, y_stdk_held_test_run, locs_held_test_run, _ = build_field_matrix_from_df_and_pred(
        df=df_held_test_run,
        y_pred_flat=y_hat_held_test_run,
    )

    locs_obs_run64 = locs_run.astype(np.float64)
    locs_held_test_run64 = locs_held_test_run.astype(np.float64)

    phi_train_unreg_run = phi_unreg_run.astype(np.float64)
    phi_train_reg_run = phi_reg_best_run.astype(np.float64)

    phi_pred_unreg_run = spatial_utils.interpolate_eigenfunction(
        locs_held_test_run64,
        locs_obs_run64,
        phi_train_unreg_run,
    )

    phi_pred_reg_run = spatial_utils.interpolate_eigenfunction(
        locs_held_test_run64,
        locs_obs_run64,
        phi_train_reg_run,
    )

    cov_unreg_run = spatial_utils.estimate_covariance(
        phi_train_unreg_run,
        R_train_obs_centered_run,
    )

    cov_reg_run = spatial_utils.estimate_covariance(
        phi_train_reg_run,
        R_train_obs_centered_run,
    )

    krig_unreg_run = spatial_utils.fixed_rank_kriging(
        phi_train_unreg_run,
        cov_unreg_run["V"],
        cov_unreg_run["eigenvalues"],
        float(cov_unreg_run["noise_var"]),
        R_test_obs_centered_run,
        phi_pred_unreg_run,
    )

    krig_reg_run = spatial_utils.fixed_rank_kriging(
        phi_train_reg_run,
        cov_reg_run["V"],
        cov_reg_run["eigenvalues"],
        float(cov_reg_run["noise_var"]),
        R_test_obs_centered_run,
        phi_pred_reg_run,
    )

    resid_held_unreg_test_run = np.asarray(
        krig_unreg_run["spatial_predictions"],
        dtype=np.float32,
    )

    resid_held_reg_test_run = np.asarray(
        krig_reg_run["spatial_predictions"],
        dtype=np.float32,
    )

    var_held_unreg_test_run = np.asarray(
        krig_unreg_run["predictive_variance"],
        dtype=np.float32,
    )

    var_held_reg_test_run = np.asarray(
        krig_reg_run["predictive_variance"],
        dtype=np.float32,
    )

    var_held_unreg_test_mat = np.broadcast_to(
        var_held_unreg_test_run.reshape(1, -1),
        resid_held_unreg_test_run.shape,
    ).astype(np.float32)

    var_held_reg_test_mat = np.broadcast_to(
        var_held_reg_test_run.reshape(1, -1),
        resid_held_reg_test_run.shape,
    ).astype(np.float32)

    y_final_held_unreg_test_run = y_stdk_held_test_run + resid_held_unreg_test_run
    y_final_held_reg_test_run = y_stdk_held_test_run + resid_held_reg_test_run

    y_true_held_test_raw_run = to_raw(y_true_held_test_run)
    y_stdk_held_test_raw_run = to_raw(y_stdk_held_test_run)
    y_final_held_unreg_test_raw_run = to_raw(y_final_held_unreg_test_run)
    y_final_held_reg_test_raw_run = to_raw(y_final_held_reg_test_run)

    sd_held_unreg_test_raw_mat = (
        np.sqrt(np.maximum(var_held_unreg_test_mat, 1e-12)).astype(np.float32) * z_sd_run
    )

    sd_held_reg_test_raw_mat = (
        np.sqrt(np.maximum(var_held_reg_test_mat, 1e-12)).astype(np.float32) * z_sd_run
    )

    z_crit = 1.96

    lower_held_unreg_test_raw_run = y_final_held_unreg_test_raw_run - z_crit * sd_held_unreg_test_raw_mat
    upper_held_unreg_test_raw_run = y_final_held_unreg_test_raw_run + z_crit * sd_held_unreg_test_raw_mat

    lower_held_reg_test_raw_run = y_final_held_reg_test_raw_run - z_crit * sd_held_reg_test_raw_mat
    upper_held_reg_test_raw_run = y_final_held_reg_test_raw_run + z_crit * sd_held_reg_test_raw_mat

    mask_held_unreg = (
        np.isfinite(y_true_held_test_raw_run)
        & np.isfinite(lower_held_unreg_test_raw_run)
        & np.isfinite(upper_held_unreg_test_raw_run)
    )

    mask_held_reg = (
        np.isfinite(y_true_held_test_raw_run)
        & np.isfinite(lower_held_reg_test_raw_run)
        & np.isfinite(upper_held_reg_test_raw_run)
    )

    heldout_stdk_test_rmse = rmse_pooled(
        y_true_held_test_raw_run,
        y_stdk_held_test_raw_run,
        np.isfinite(y_true_held_test_raw_run) & np.isfinite(y_stdk_held_test_raw_run),
    )

    heldout_unreg_test_rmse = rmse_pooled(
        y_true_held_test_raw_run,
        y_final_held_unreg_test_raw_run,
        np.isfinite(y_true_held_test_raw_run) & np.isfinite(y_final_held_unreg_test_raw_run),
    )

    heldout_reg_best_test_rmse = rmse_pooled(
        y_true_held_test_raw_run,
        y_final_held_reg_test_raw_run,
        np.isfinite(y_true_held_test_raw_run) & np.isfinite(y_final_held_reg_test_raw_run),
    )

    heldout_unreg_test_mpiw = mpiw(
        lower_held_unreg_test_raw_run,
        upper_held_unreg_test_raw_run,
        mask_held_unreg,
    )

    heldout_reg_best_test_mpiw = mpiw(
        lower_held_reg_test_raw_run,
        upper_held_reg_test_raw_run,
        mask_held_reg,
    )

    heldout_unreg_test_cp = cp_percent(
        y_true_held_test_raw_run,
        lower_held_unreg_test_raw_run,
        upper_held_unreg_test_raw_run,
        mask_held_unreg,
    )

    heldout_reg_best_test_cp = cp_percent(
        y_true_held_test_raw_run,
        lower_held_reg_test_raw_run,
        upper_held_reg_test_raw_run,
        mask_held_reg,
    )

    def collect_metrics(prefix: str, y_pred_std: np.ndarray):
        return {
            f"{prefix}_rmse_train": rmse_std(y_pred_std, train_time_idx_run),
            f"{prefix}_rmse_val": rmse_std(y_pred_std, val_time_idx_run),
            f"{prefix}_rmse_test": rmse_std(y_pred_std, test_time_idx_run),
            f"{prefix}_rmse_full": rmse_std(y_pred_std, full_time_idx_run),
            f"{prefix}_rmse_train_raw": rmse_raw(y_pred_std, train_time_idx_run),
            f"{prefix}_rmse_val_raw": rmse_raw(y_pred_std, val_time_idx_run),
            f"{prefix}_rmse_test_raw": rmse_raw(y_pred_std, test_time_idx_run),
            f"{prefix}_rmse_full_raw": rmse_raw(y_pred_std, full_time_idx_run),
            f"{prefix}_covfrob_train": covfrob_std(y_pred_std, train_time_idx_run),
            f"{prefix}_covfrob_val": covfrob_std(y_pred_std, val_time_idx_run),
            f"{prefix}_covfrob_test": covfrob_std(y_pred_std, test_time_idx_run),
            f"{prefix}_covfrob_full": covfrob_std(y_pred_std, full_time_idx_run),
            f"{prefix}_covfrob_train_raw": covfrob_raw(y_pred_std, train_time_idx_run),
            f"{prefix}_covfrob_val_raw": covfrob_raw(y_pred_std, val_time_idx_run),
            f"{prefix}_covfrob_test_raw": covfrob_raw(y_pred_std, test_time_idx_run),
            f"{prefix}_covfrob_full_raw": covfrob_raw(y_pred_std, full_time_idx_run),
            f"{prefix}_sv_loss_train": sv_loss_std(y_pred_std, train_time_idx_run),
            f"{prefix}_sv_loss_val": sv_loss_std(y_pred_std, val_time_idx_run),
            f"{prefix}_sv_loss_test": sv_loss_std(y_pred_std, test_time_idx_run),
            f"{prefix}_sv_loss_full": sv_loss_std(y_pred_std, full_time_idx_run),
            f"{prefix}_sv_loss_train_raw": sv_loss_raw(y_pred_std, train_time_idx_run),
            f"{prefix}_sv_loss_val_raw": sv_loss_raw(y_pred_std, val_time_idx_run),
            f"{prefix}_sv_loss_test_raw": sv_loss_raw(y_pred_std, test_time_idx_run),
            f"{prefix}_sv_loss_full_raw": sv_loss_raw(y_pred_std, full_time_idx_run),
        }

    summary_row = {
        "seed": int(run_seed),
        "n_sites_full": int(n_sites_full_run),
        "n_sites_keep": int(len(keep_sites_run)),
        "n_sites_observed": int(len(observed_sites_run)),
        "n_sites_heldout": int(len(heldout_sites_run)),
        "n_times": int(n_times_run),
        "n_train_times": int(len(train_time_idx_run)),
        "n_val_times": int(len(val_time_idx_run)),
        "n_test_times": int(len(test_time_idx_run)),
        "z_mean_train_raw": float(z_mean_run),
        "z_sd_train_raw": float(z_sd_run),
        "best_tau1": float(best_tau1_run),
        "best_tau2": float(best_tau2_run),
        "best_trial": int(best_trial_no),
        objective_label(): float(best_objective_run),
        "heldout_stdk_test_rmse": float(heldout_stdk_test_rmse),
        "heldout_unreg_test_rmse": float(heldout_unreg_test_rmse),
        "heldout_reg_best_test_rmse": float(heldout_reg_best_test_rmse),
        "heldout_unreg_test_mpiw": float(heldout_unreg_test_mpiw),
        "heldout_reg_best_test_mpiw": float(heldout_reg_best_test_mpiw),
        "heldout_unreg_test_cp": float(heldout_unreg_test_cp),
        "heldout_reg_best_test_cp": float(heldout_reg_best_test_cp),
        "unreg_recon_mse_train": float(diag_train_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_train": float(diag_train_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_train": float(diag_train_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_train": float(diag_train_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_train": float(diag_train_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_train": float(diag_train_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_train": float(diag_train_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_train": float(diag_train_unreg_run["l1_over_recon"]),
        "unreg_recon_mse_all": float(diag_all_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_all": float(diag_all_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_all": float(diag_all_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_all": float(diag_all_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_all": float(diag_all_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_all": float(diag_all_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_all": float(diag_all_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_all": float(diag_all_unreg_run["l1_over_recon"]),
        "reg_best_recon_mse_train": float(diag_train_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_train": float(diag_train_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_train": float(diag_train_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_train": float(diag_train_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_train": float(diag_train_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_train": float(diag_train_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_train": float(diag_train_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_train": float(diag_train_reg_best_run["l1_over_recon"]),
        "reg_best_recon_mse_all": float(diag_all_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_all": float(diag_all_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_all": float(diag_all_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_all": float(diag_all_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_all": float(diag_all_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_all": float(diag_all_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_all": float(diag_all_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_all": float(diag_all_reg_best_run["l1_over_recon"]),
    }

    summary_row.update(collect_metrics("stdk", y_stdk_run))
    summary_row.update(collect_metrics("unreg", y_final_unreg_run))
    summary_row.update(collect_metrics("reg_best", y_final_reg_best_run))

    np.savez_compressed(
        PRED_DIR / f"seed_{run_seed}.npz",
        seed=np.array([run_seed], dtype=np.int32),
        keep_sites_run=np.asarray(keep_sites_run, dtype=np.int32),
        observed_sites_run=np.asarray(observed_sites_run, dtype=np.int32),
        heldout_sites_run=np.asarray(heldout_sites_run, dtype=np.int32),
        n_sites_full_run=np.array([n_sites_full_run], dtype=np.int32),
        uniq_t_run=np.asarray(uniq_t_run, dtype=np.float32),
        train_time_idx_run=np.asarray(train_time_idx_run, dtype=np.int32),
        val_time_idx_run=np.asarray(val_time_idx_run, dtype=np.int32),
        test_time_idx_run=np.asarray(test_time_idx_run, dtype=np.int32),
        locs_run=locs_run.astype(np.float32),
        y_true_run=y_true_run.astype(np.float32),
        y_true_raw_run=y_true_raw_run.astype(np.float32),
        y_stdk_run=y_stdk_run.astype(np.float32),
        y_stdk_raw_run=y_stdk_raw_run.astype(np.float32),
        y_final_unreg_run=y_final_unreg_run.astype(np.float32),
        y_final_unreg_raw_run=y_final_unreg_raw_run.astype(np.float32),
        y_final_reg_best_run=y_final_reg_best_run.astype(np.float32),
        y_final_reg_best_raw_run=y_final_reg_best_raw_run.astype(np.float32),
        residual_full_unreg_run=residual_full_unreg_run.astype(np.float32),
        residual_full_reg_best_run=residual_full_reg_best_run.astype(np.float32),
        locs_held_test_run=locs_held_test_run.astype(np.float32),
        y_true_held_test_run=y_true_held_test_run.astype(np.float32),
        y_true_held_test_raw_run=y_true_held_test_raw_run.astype(np.float32),
        y_stdk_held_test_run=y_stdk_held_test_run.astype(np.float32),
        y_stdk_held_test_raw_run=y_stdk_held_test_raw_run.astype(np.float32),
        y_final_held_unreg_test_run=y_final_held_unreg_test_run.astype(np.float32),
        y_final_held_unreg_test_raw_run=y_final_held_unreg_test_raw_run.astype(np.float32),
        y_final_held_reg_test_run=y_final_held_reg_test_run.astype(np.float32),
        y_final_held_reg_test_raw_run=y_final_held_reg_test_raw_run.astype(np.float32),
        var_held_unreg_test_run=var_held_unreg_test_run.astype(np.float32),
        var_held_reg_test_run=var_held_reg_test_run.astype(np.float32),
        lower_held_unreg_test_raw_run=lower_held_unreg_test_raw_run.astype(np.float32),
        upper_held_unreg_test_raw_run=upper_held_unreg_test_raw_run.astype(np.float32),
        lower_held_reg_test_raw_run=lower_held_reg_test_raw_run.astype(np.float32),
        upper_held_reg_test_raw_run=upper_held_reg_test_raw_run.astype(np.float32),
        z_mean_run=np.array([z_mean_run], dtype=np.float32),
        z_sd_run=np.array([z_sd_run], dtype=np.float32),
        best_tau1=np.array([best_tau1_run], dtype=np.float64),
        best_tau2=np.array([best_tau2_run], dtype=np.float64),
        best_trial=np.array([best_trial_no], dtype=np.int32),
    )

    print(
        f"[seed {run_seed}] best trial={best_trial_no} | "
        f"tau1={best_tau1_run:.3e} | tau2={best_tau2_run:.3e} | "
        f"test_rmse={summary_row['reg_best_rmse_test_raw']:.6f} | "
        f"test_covfrob={summary_row['reg_best_covfrob_test']:.6f} | "
        f"test_sv={summary_row['reg_best_sv_loss_test']:.6f} | "
        f"heldout_test_rmse={summary_row['heldout_reg_best_test_rmse']:.6f} | "
        f"heldout_mpiw={summary_row['heldout_reg_best_test_mpiw']:.6f} | "
        f"heldout_cp={summary_row['heldout_reg_best_test_cp']:.2f}",
        flush=True,
    )

    return summary_row

## run

In [ ]:
rows_fixed_k = []

for r in range(N_RUNS_FIXED_K):
    run_seed = SEED + r * 1000
    print(
        f"\n===== FIXED K RUN {r + 1}/{N_RUNS_FIXED_K} | "
        f"seed={run_seed} | target={TUNING_TARGET} | "
        f"space_keep={SPACE_RATIO_KEEP} | "
        f"heldout_station_ratio={HELDOUT_STATION_RATIO} | "
        f"time_train={TRAIN_RATIO_TIME} | "
        f"time_val={VAL_RATIO_TIME} | "
        f"time_test={TEST_RATIO_TIME} =====",
        flush=True,
    )
    rows_fixed_k.append(run_once_fixed_k(run_seed))

results_fixed_k_df = pd.DataFrame(rows_fixed_k)
results_fixed_k_df.to_csv(SUMMARY_CSV, index=False)

trial_files = sorted(TRIAL_DIR.glob("seed_*_trials.csv"))
if len(trial_files) > 0:
    all_trial_df = pd.concat(
        [pd.read_csv(f) for f in trial_files],
        ignore_index=True,
    )
    all_trial_df.to_csv(ALL_TRIAL_CSV, index=False)
else:
    all_trial_df = pd.DataFrame()

print("\n=== Summary saved ===", flush=True)
print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)
print("ALL_TRIAL_CSV:", ALL_TRIAL_CSV, flush=True)
print("n_summary_rows:", len(results_fixed_k_df), flush=True)
print("n_trial_rows:", len(all_trial_df), flush=True)

## summary

In [ ]:
summary_df = load_summary_or_empty()

if summary_df.empty:
    print("No summary file found.", flush=True)
else:
    print("\n=== Main paths ===", flush=True)
    print_main_paths()

    print("\n=== RMSE summary (standardized scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_rmse_{split}'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_rmse_{split}'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_rmse_{split}'])}",
            flush=True,
        )

    print("\n=== RMSE summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_rmse_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_rmse_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_rmse_{split}_raw'])}",
            flush=True,
        )

    print("\n=== CovFrob summary (standardized scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_covfrob_{split}'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_covfrob_{split}'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_covfrob_{split}'])}",
            flush=True,
        )

    print("\n=== CovFrob summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_covfrob_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_covfrob_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_covfrob_{split}_raw'])}",
            flush=True,
        )

    print("\n=== Semivariogram matching loss summary (standardized scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_sv_loss_{split}'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_sv_loss_{split}'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_sv_loss_{split}'])}",
            flush=True,
        )

    print("\n=== Semivariogram matching loss summary (raw scale) ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(summary_df[f'stdk_sv_loss_{split}_raw'])} | "
            f"unreg = {fmt_pm(summary_df[f'unreg_sv_loss_{split}_raw'])} | "
            f"reg_best = {fmt_pm(summary_df[f'reg_best_sv_loss_{split}_raw'])}",
            flush=True,
        )

    print("\n=== Held-out station TEST RMSE summary ===", flush=True)
    print(
        f"{'test':>5} | "
        f"stdk = {fmt_pm(summary_df['heldout_stdk_test_rmse'])} | "
        f"unreg = {fmt_pm(summary_df['heldout_unreg_test_rmse'])} | "
        f"reg_best = {fmt_pm(summary_df['heldout_reg_best_test_rmse'])}",
        flush=True,
    )

    print("\n=== Held-out station TEST MPIW summary ===", flush=True)
    print(
        f"{'test':>5} | "
        f"stdk = n/a | "
        f"unreg = {fmt_pm(summary_df['heldout_unreg_test_mpiw'])} | "
        f"reg_best = {fmt_pm(summary_df['heldout_reg_best_test_mpiw'])}",
        flush=True,
    )

    print("\n=== Held-out station TEST CP(%) summary ===", flush=True)
    print(
        f"{'test':>5} | "
        f"stdk = n/a | "
        f"unreg = {fmt_pm(summary_df['heldout_unreg_test_cp'])} | "
        f"reg_best = {fmt_pm(summary_df['heldout_reg_best_test_cp'])}",
        flush=True,
    )

    print("\n=== Held-out station count summary ===", flush=True)
    print(f"n_sites_observed: {fmt_pm(summary_df['n_sites_observed'])}", flush=True)
    print(f"n_sites_heldout : {fmt_pm(summary_df['n_sites_heldout'])}", flush=True)

    print("\n=== Best tuning summary ===", flush=True)
    print(f"best_tau1      : {fmt_pm(summary_df['best_tau1'])}", flush=True)
    print(f"best_tau2      : {fmt_pm(summary_df['best_tau2'])}", flush=True)
    print(f"{objective_label():<15}: {fmt_pm(summary_df[objective_label()])}", flush=True)

## diagnostics

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("./weather2k")
TUNING_TARGET = "rmse"
TARGET_VAR_IDX = 4
T_KEEP = 1000
K_FIXED = 40
SPACE_RATIO_KEEP = 0.1
TRAIN_RATIO_TIME = 0.1
VAL_RATIO_TIME = 0.1
TEST_RATIO_TIME = 0.8

REPEAT_DIR = RESULT_DIR / (
    f"{TUNING_TARGET}_tuning"
    f"/var{TARGET_VAR_IDX}_tkeep{T_KEEP}"
    f"_k_{K_FIXED}_fixedspace{SPACE_RATIO_KEEP}"
    f"_time_train{TRAIN_RATIO_TIME}_val{VAL_RATIO_TIME}_test{TEST_RATIO_TIME}"
)
PRED_DIR = REPEAT_DIR / "saved_predictions"

print("REPEAT_DIR =", REPEAT_DIR)
print("PRED_DIR   =", PRED_DIR)

def pooled_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(mask):
        return np.nan
    err = y_true[mask] - y_pred[mask]
    return float(np.sqrt(np.mean(err ** 2)))

def cp_percent(y_true, lower, upper):
    y_true = np.asarray(y_true, dtype=np.float64)
    lower = np.asarray(lower, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)
    mask = np.isfinite(y_true) & np.isfinite(lower) & np.isfinite(upper)
    if not np.any(mask):
        return np.nan
    covered = (y_true[mask] >= lower[mask]) & (y_true[mask] <= upper[mask])
    return float(np.mean(covered) * 100.0)

def mpiw(lower, upper):
    lower = np.asarray(lower, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)
    mask = np.isfinite(lower) & np.isfinite(upper)
    if not np.any(mask):
        return np.nan
    return float(np.mean(upper[mask] - lower[mask]))

def make_timewise_rows(seed, method, y_true, y_pred, lower, upper):
    rows = []
    T = y_true.shape[0]
    for t in range(T):
        rows.append({
            "seed": seed,
            "method": method,
            "time_idx_local": t,
            "n_sites": int(np.isfinite(y_true[t]).sum()),
            "rmse": pooled_rmse(y_true[t], y_pred[t]),
            "cp_percent": cp_percent(y_true[t], lower[t], upper[t]),
            "mpiw": mpiw(lower[t], upper[t]),
        })
    return rows

def make_sitewise_stdres_rows(seed, method, y_true, y_pred, var_raw_1d):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    var_raw_1d = np.asarray(var_raw_1d, dtype=np.float64)

    sd_1d = np.sqrt(np.maximum(var_raw_1d, 1e-12))
    rows = []

    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        mask = np.isfinite(yt) & np.isfinite(yp)

        if not np.any(mask):
            rows.append({
                "seed": seed,
                "method": method,
                "site_idx_local": j,
                "n_times": 0,
                "rmse": np.nan,
                "pred_sd": float(sd_1d[j]),
                "std_resid_mean": np.nan,
                "std_resid_sd": np.nan,
                "std_resid_rmse": np.nan,
                "abs_std_resid_mean": np.nan,
                "cp_percent": np.nan,
            })
            continue

        resid = yt[mask] - yp[mask]
        z = resid / sd_1d[j]

        rows.append({
            "seed": seed,
            "method": method,
            "site_idx_local": j,
            "n_times": int(mask.sum()),
            "rmse": float(np.sqrt(np.mean(resid ** 2))),
            "pred_sd": float(sd_1d[j]),
            "std_resid_mean": float(np.mean(z)),
            "std_resid_sd": float(np.std(z, ddof=0)),
            "std_resid_rmse": float(np.sqrt(np.mean(z ** 2))),
            "abs_std_resid_mean": float(np.mean(np.abs(z))),
            "cp_percent": float(np.mean(np.abs(z) <= 1.96) * 100.0),
        })

    return rows

npz_files = sorted(PRED_DIR.glob("seed_*.npz"))
if len(npz_files) == 0:
    raise FileNotFoundError(f"No seed_*.npz files found in {PRED_DIR}")

overall_rows = []
timewise_rows = []
sitewise_rows = []

for f in npz_files:
    arr = np.load(f)

    try:
        seed = int(f.stem.split("_")[1])
    except Exception:
        seed = -1

    y_true_raw = arr["y_true_held_test_raw_run"].astype(np.float64)

    y_stdk_raw = arr["y_stdk_held_test_raw_run"].astype(np.float64)
    y_unreg_raw = arr["y_final_held_unreg_test_raw_run"].astype(np.float64)
    y_reg_raw = arr["y_final_held_reg_test_raw_run"].astype(np.float64)

    lower_unreg_raw = arr["lower_held_unreg_test_raw_run"].astype(np.float64)
    upper_unreg_raw = arr["upper_held_unreg_test_raw_run"].astype(np.float64)
    lower_reg_raw = arr["lower_held_reg_test_raw_run"].astype(np.float64)
    upper_reg_raw = arr["upper_held_reg_test_raw_run"].astype(np.float64)

    var_unreg_std = arr["var_held_unreg_test_run"].astype(np.float64)
    var_reg_std = arr["var_held_reg_test_run"].astype(np.float64)
    z_sd_run = float(arr["z_sd_run"][0])

    var_unreg_raw = var_unreg_std * (z_sd_run ** 2)
    var_reg_raw = var_reg_std * (z_sd_run ** 2)

    best_tau1 = float(arr["best_tau1"][0]) if "best_tau1" in arr else np.nan
    best_tau2 = float(arr["best_tau2"][0]) if "best_tau2" in arr else np.nan
    best_trial = int(arr["best_trial"][0]) if "best_trial" in arr else -1

    overall_rows.extend([
        {
            "seed": seed,
            "method": "stdk",
            "best_tau1": best_tau1,
            "best_tau2": best_tau2,
            "best_trial": best_trial,
            "rmse": pooled_rmse(y_true_raw, y_stdk_raw),
            "cp_percent": np.nan,
            "mpiw": np.nan,
        },
        {
            "seed": seed,
            "method": "unreg",
            "best_tau1": 0.0,
            "best_tau2": 0.0,
            "best_trial": -1,
            "rmse": pooled_rmse(y_true_raw, y_unreg_raw),
            "cp_percent": cp_percent(y_true_raw, lower_unreg_raw, upper_unreg_raw),
            "mpiw": mpiw(lower_unreg_raw, upper_unreg_raw),
        },
        {
            "seed": seed,
            "method": "reg_best",
            "best_tau1": best_tau1,
            "best_tau2": best_tau2,
            "best_trial": best_trial,
            "rmse": pooled_rmse(y_true_raw, y_reg_raw),
            "cp_percent": cp_percent(y_true_raw, lower_reg_raw, upper_reg_raw),
            "mpiw": mpiw(lower_reg_raw, upper_reg_raw),
        },
    ])

    timewise_rows.extend(
        make_timewise_rows(seed, "unreg", y_true_raw, y_unreg_raw, lower_unreg_raw, upper_unreg_raw)
    )
    timewise_rows.extend(
        make_timewise_rows(seed, "reg_best", y_true_raw, y_reg_raw, lower_reg_raw, upper_reg_raw)
    )

    sitewise_rows.extend(
        make_sitewise_stdres_rows(seed, "unreg", y_true_raw, y_unreg_raw, var_unreg_raw)
    )
    sitewise_rows.extend(
        make_sitewise_stdres_rows(seed, "reg_best", y_true_raw, y_reg_raw, var_reg_raw)
    )

overall_df = pd.DataFrame(overall_rows)
timewise_df = pd.DataFrame(timewise_rows)
sitewise_df = pd.DataFrame(sitewise_rows)

print("\n=== Overall held-out diagnostics by seed ===")
print(overall_df.sort_values(["seed", "method"]).to_string(index=False))

print("\n=== Overall summary across seeds ===")
overall_summary = (
    overall_df
    .groupby("method", as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_sd=("rmse", "std"),
        cp_mean=("cp_percent", "mean"),
        cp_sd=("cp_percent", "std"),
        mpiw_mean=("mpiw", "mean"),
        mpiw_sd=("mpiw", "std"),
    )
)
print(overall_summary.to_string(index=False))

print("\n=== Timewise summary across seeds ===")
timewise_summary = (
    timewise_df
    .groupby("method", as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_sd=("rmse", "std"),
        cp_mean=("cp_percent", "mean"),
        cp_sd=("cp_percent", "std"),
        mpiw_mean=("mpiw", "mean"),
        mpiw_sd=("mpiw", "std"),
    )
)
print(timewise_summary.to_string(index=False))

print("\n=== Sitewise standardized residual summary across seeds ===")
sitewise_summary = (
    sitewise_df
    .groupby("method", as_index=False)
    .agg(
        pred_sd_mean=("pred_sd", "mean"),
        rmse_mean=("rmse", "mean"),
        std_resid_mean_mean=("std_resid_mean", "mean"),
        std_resid_sd_mean=("std_resid_sd", "mean"),
        std_resid_rmse_mean=("std_resid_rmse", "mean"),
        abs_std_resid_mean_mean=("abs_std_resid_mean", "mean"),
        cp_mean=("cp_percent", "mean"),
    )
)
print(sitewise_summary.to_string(index=False))

print("\n=== Worst 10 reg_best time points by CP ===")
print(
    timewise_df[timewise_df["method"] == "reg_best"]
    .sort_values(["cp_percent", "rmse"], ascending=[True, False])
    .head(10)
    .to_string(index=False)
)

print("\n=== Worst 10 reg_best sites by standardized residual RMSE ===")
print(
    sitewise_df[sitewise_df["method"] == "reg_best"]
    .sort_values(["std_resid_rmse", "cp_percent"], ascending=[False, True])
    .head(10)
    .to_string(index=False)
)